# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides users through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR^2):

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print('Dataset Name:')
print(metadata['name'])
print('\nDataset Description:')
print(metadata['description'])
print('\nLicense:', metadata.get('license', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities in the dataset are referenced by their `@id` fields.

In [ ]:
# Retrieve available record sets with their @id
croissant_schema = dataset.metadata.to_json()
record_sets = croissant_schema.get('recordSet', []) # Might be empty if recordSet is not populated
if not record_sets:
    print('No record sets found in metadata. Attempting to discover from package...')
    # mlcroissant will expose recordSet via dataset.record_sets()
    record_sets = dataset.record_sets()
else:
    record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]

print('Record Sets available:')
for rs in record_sets:
    print(f' - {rs}')

# Explore available fields and columns for each record set
for rs in record_sets:
    print(f'\nFields and columns for Record Set {rs}:')
    fields = dataset.fields(record_set=rs)
    for field in fields:
        print(f"  Field @id: {field['@id']}  |  Name: {field.get('name','')}")
        if 'column' in field:
            for col in field['column']:
                if isinstance(col, dict):
                    print(f"    Column @id: {col.get('@id', '')}  |  Name: {col.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

_Note_: All entities are referenced by their `@id` field. The following example assumes at least one accessible record set is available via `@id`.

In [ ]:
# List of discovered record set @ids
record_sets_ids = record_sets if record_sets else dataset.record_sets()

dataframes = {}

for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded DataFrame for Record Set @id '{rs_id}':")
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for Record Set @id '{rs_id}'.")

# Choose primary record set for further analysis
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    df = dataframes[primary_record_set_id]
else:
    primary_record_set_id = None
    df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data by key attributes. All entities referenced by their `@id`.

In [ ]:
# Example EDA: Remove outliers, normalize, group
# Select a numeric field for analysis

if not df.empty:
    # Find numeric columns
    numeric_cols = df.select_dtypes(include=[float, int]).columns.tolist()
    if not numeric_cols:
        print('No numeric fields found for EDA.')
    else:
        # Use the first numeric column for demonstration
        numeric_field_id = numeric_cols[0]

        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        print(f"\nFiltering records where field '{numeric_field_id}' > {threshold:.3f}")
        filtered_df = df[df[numeric_field_id] > threshold]
        print(filtered_df.head())

        # Normalize the numeric field (Z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field, if any
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if cat_cols:
            group_field_id = cat_cols[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print('No categorical fields available for grouping.')
else:
    print('No valid DataFrame available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Reference fields and columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Example: Histogram of numeric field
if not df.empty and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Example: Boxplot grouped by categorical field
if not df.empty and 'group_field_id' in locals() and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df.dropna(subset=[numeric_field_id, group_field_id]).boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"Boxplot of '{numeric_field_id}' grouped by '{group_field_id}'")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides valuable insight into adoption predictors for indigenous and modern rangeland management practices in Northern Kenya.
- Key fields and their `@id`s facilitate reproducible, structured analysis.
- Exploratory analysis demonstrates filtering, normalization, and grouping, enabling policy-relevant insights and academic study.
- Dataset limitations and biases are documented, supporting fair, transparent use.